# 🔍 Explicabilidade do Modelo - XGBoost Final

## Objetivo

Explicar **como** o modelo XGBoost final toma decisões ao prever se um FII superará o IFIX nos próximos 7 pregões.

---

## ⚠️ Princípios de Governança

1. **Não modificar o modelo final**: Este notebook apenas explica o modelo já validado
2. **Não usar explicabilidade para retreinar**: Explicações não implicam melhorias automáticas
3. **Não afirmar causalidade**: SHAP e importância medem influência nas previsões, não causalidade econômica
4. **Preservar congelamento**: Mesmas features, hiperparâmetros, seed e dados do notebook 40

---

## 📋 Estrutura

1. Reconstrução controlada do modelo
2. Importância global nativa (XGBoost)
3. Explicabilidade global (SHAP)
4. Dependência das principais features
5. Explicabilidade por ticker
6. Explicações locais
7. Análise de erros
8. Análise do ranking Top 1
9. Variáveis macroeconômicas
10. Limitações
11. Conclusões

---

In [0]:
%pip install xgboost scikit-learn shap matplotlib seaborn
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Seed (mesmo do notebook 40)
SEED = 42
np.random.seed(SEED)

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("="*80)
print("🔍 EXPLICABILIDADE DO MODELO XGBOOST FINAL")
print("="*80)
print(f"\nSeed: {SEED}")
print("\nObjetivo: Explicar como o modelo toma decisões")
print("Abordagem: Importância nativa + SHAP TreeExplainer")
print("Princípio: Explicações não implicam causalidade")
print("="*80)

In [0]:
print("="*80)
print("📂 1. RECONSTRUÇÃO CONTROLADA DO MODELO")
print("="*80)

print("\n⚠️ Reconstruindo EXATAMENTE o mesmo modelo do notebook 40...\n")

# Carregar Gold V1
df = spark.table("workspace.gold.fii_features_v1").toPandas()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

print(f"✅ Gold V1 carregada: {df.shape[0]:,} registros")
print(f"Período: {df['date'].min().date()} a {df['date'].max().date()}")
print(f"Tickers: {sorted(df['ticker'].unique())}")

# Features (idênticas ao notebook 40)
features = [
    'close', 'volume', 'has_trading',
    'return_1d', 'return_7d', 'return_30d', 'return_90d',
    'ifix_return_1d', 'ifix_return_7d', 'ifix_return_30d', 'ifix_return_90d',
    'volatility_30d', 'volatility_90d',
    'dividend_yield_12m', 'dividend_history_days', 'days_since_last_dividend', 'has_dividend_today',
    'alpha_30d', 'alpha_90d',
    'selic', 'ipca', 'dolar', 'desemprego'
]

print(f"\n📊 Features: {len(features)}")
print(f"Lista: {features}")

In [0]:
# Período de holdout (idêntico ao notebook 40)
holdout_start = pd.to_datetime('2025-03-01')
holdout_end = pd.to_datetime('2026-02-28')

# Encontrar última data de treino (7 pregões antes do holdout - purge gap)
dates_before_holdout = sorted(df[df['date'] < holdout_start]['date'].unique())
last_train_date = dates_before_holdout[-8]  # -8 para gap de 7 pregões

print(f"\n📅 Períodos:")
print(f"  Holdout: {holdout_start.date()} a {holdout_end.date()}")
print(f"  Última data de treino: {last_train_date.date()}")
print(f"  Purge gap: 7 pregões")

# Dividir treino/holdout
train_df = df[df['date'] <= last_train_date].copy()
holdout_df = df[(df['date'] >= holdout_start) & (df['date'] <= holdout_end)].copy()

print(f"\n📊 Tamanhos:")
print(f"  Treino: {len(train_df):,} registros ({train_df['date'].min().date()} a {train_df['date'].max().date()})")
print(f"  Holdout: {len(holdout_df):,} registros ({holdout_df['date'].min().date()} a {holdout_df['date'].max().date()})")

# Preparar X, y
X_train = train_df[features].values
y_train = train_df['target_7d'].values
X_holdout = holdout_df[features].values
y_holdout = holdout_df['target_7d'].values

print(f"\n✅ Dados preparados:")
print(f"  X_train: {X_train.shape}")
print(f"  X_holdout: {X_holdout.shape}")

In [0]:
# Hiperparâmetros CONGELADOS (idênticos ao notebook 40)
FINAL_HYPERPARAMETERS = {
    'n_estimators': 150,
    'max_depth': 6,
    'learning_rate': 0.07,
    'min_child_weight': 1,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
    'gamma': 0.0,
    'reg_alpha': 0.01,
    'reg_lambda': 1.5,
    'early_stopping_rounds': 20,
    'random_state': SEED,
    'n_jobs': -1,
    'tree_method': 'hist'
}

print("\n🔧 Hiperparâmetros congelados (do notebook 40):")
for k, v in FINAL_HYPERPARAMETERS.items():
    print(f"  {k}: {v}")

# Treinar modelo
print("\n🎯 Treinando modelo...")
model = xgb.XGBClassifier(**FINAL_HYPERPARAMETERS)
model.fit(X_train, y_train, eval_set=[(X_train, y_train)], verbose=False)

print(f"\n✅ Modelo treinado (seed={SEED})")

# Predições no holdout
y_pred_proba = model.predict_proba(X_holdout)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

print(f"\n🔮 Previsões geradas: {len(y_pred_proba):,}")

In [0]:
# Calcular métricas
roc_auc = roc_auc_score(y_holdout, y_pred_proba)
accuracy = accuracy_score(y_holdout, y_pred)
precision = precision_score(y_holdout, y_pred, zero_division=0)
recall = recall_score(y_holdout, y_pred, zero_division=0)
f1 = f1_score(y_holdout, y_pred, zero_division=0)

print("\n" + "="*80)
print("✅ VALIDAÇÃO: COMPARAÇÃO COM NOTEBOOK 40")
print("="*80)

# Métricas esperadas do notebook 40
expected = {
    'ROC-AUC': 0.6175,
    'Accuracy': 0.5830,
    'Precision': 0.5734,
    'Recall': 0.5648,
    'F1': 0.5690
}

obtained = {
    'ROC-AUC': roc_auc,
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1': f1
}

print("\n📈 Comparação de Métricas:\n")
print(f"{'Métrica':<15} {'Esperado':>12} {'Obtido':>12} {'Diferença':>12} {'Status':>10}")
print("-" * 65)

for metric in expected:
    exp = expected[metric]
    obt = obtained[metric]
    diff = obt - exp
    status = "✅ OK" if abs(diff) < 0.001 else "⚠️ DIFF"
    print(f"{metric:<15} {exp:>12.4f} {obt:>12.4f} {diff:>12.4f} {status:>10}")

print("\n✅ Modelo reconstruído com sucesso!")
print("Diferenças devem ser zero ou explicadas por precisão numérica.")

In [0]:
print("\n" + "="*80)
print("📊 2. IMPORTÂNCIA GLOBAL NATIVA (XGBOOST)")
print("="*80)

print("\n⚠️ Importância nativa mede a utilização das features pelo modelo, NÃO causalidade.\n")

# Obter importâncias
importances = {
    'gain': model.get_booster().get_score(importance_type='gain'),
    'weight': model.get_booster().get_score(importance_type='weight'),
    'cover': model.get_booster().get_score(importance_type='cover')
}

# Criar DataFrame
importance_df = pd.DataFrame({
    'feature': features,
    'gain': [importances['gain'].get(f'f{i}', 0) for i in range(len(features))],
    'weight': [importances['weight'].get(f'f{i}', 0) for i in range(len(features))],
    'cover': [importances['cover'].get(f'f{i}', 0) for i in range(len(features))]
})

# Normalizar
importance_df['gain_norm'] = importance_df['gain'] / importance_df['gain'].sum() * 100
importance_df['weight_norm'] = importance_df['weight'] / importance_df['weight'].sum() * 100
importance_df['cover_norm'] = importance_df['cover'] / importance_df['cover'].sum() * 100

# Ordenar por gain
importance_df = importance_df.sort_values('gain', ascending=False).reset_index(drop=True)

print("🏆 Top 10 Features (por Gain):\n")
print(importance_df[['feature', 'gain_norm', 'weight_norm', 'cover_norm']].head(10).to_string(index=False))

print(f"\n\n📈 Ranking Completo (por Gain):\n")
for i, row in importance_df.iterrows():
    print(f"{i+1:2d}. {row['feature']:<25} Gain: {row['gain_norm']:6.2f}%  Weight: {row['weight_norm']:6.2f}%  Cover: {row['cover_norm']:6.2f}%")

In [0]:
# Agrupar por tipo de feature
print("\n" + "="*80)
print("📋 Importância por Tipo de Feature")
print("="*80)

macro_features = ['selic', 'ipca', 'dolar', 'desemprego']
ifix_features = ['ifix_return_1d', 'ifix_return_7d', 'ifix_return_30d', 'ifix_return_90d']
return_features = ['return_1d', 'return_7d', 'return_30d', 'return_90d']
dividend_features = ['dividend_yield_12m', 'dividend_history_days', 'days_since_last_dividend', 'has_dividend_today']
volatility_features = ['volatility_30d', 'volatility_90d', 'alpha_30d', 'alpha_90d']

groups = {
    'Macro': macro_features,
    'Retornos IFIX': ifix_features,
    'Retornos FII': return_features,
    'Dividendos': dividend_features,
    'Volatilidade/Alpha': volatility_features
}

for group_name, group_features in groups.items():
    group_df = importance_df[importance_df['feature'].isin(group_features)]
    total_gain = group_df['gain_norm'].sum()
    print(f"\n📊 {group_name}: {total_gain:.2f}% (gain total)")
    for _, row in group_df.iterrows():
        print(f"  - {row['feature']:<25} {row['gain_norm']:6.2f}%")

In [0]:
print("\n" + "="*80)
print("🔮 3. EXPLICABILIDADE GLOBAL (SHAP)")
print("="*80)

print("\n⚠️ SHAP explica influência nas previsões, NÃO causalidade econômica.\n")

# Usar amostra do holdout se for muito pesado (ou holdout completo se viável)
# Vamos tentar holdout completo primeiro
print(f"📋 Calculando valores SHAP no holdout ({len(X_holdout):,} observações)...")
print("Isso pode levar alguns minutos...\n")

# SHAP TreeExplainer (otimizado para XGBoost)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_holdout)

print("✅ Valores SHAP calculados!")
print(f"Shape: {shap_values.shape}")
print(f"Base value (valor esperado): {explainer.expected_value:.4f}")

In [0]:
# SHAP Summary Plot (beeswarm)
print("\n🐝 Gerando SHAP Summary Plot (beeswarm)...")
fig, ax = plt.subplots(figsize=(12, 10))
shap.summary_plot(shap_values, X_holdout, feature_names=features, show=False)
plt.title('SHAP Summary Plot - Distribuição de Impactos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Summary plot gerado!")
print("\n📄 Interpretação:")
print("  - Eixo Y: features ordenadas por importância")
print("  - Eixo X: impacto SHAP (positivo = aumenta proba, negativo = reduz proba)")
print("  - Cor: valor da feature (vermelho = alto, azul = baixo)")
print("  - Cada ponto = uma observação do holdout")

In [0]:
# SHAP Bar Plot (importância média absoluta)
print("\n📊 Gerando SHAP Bar Plot (importância média absoluta)...")
fig, ax = plt.subplots(figsize=(12, 10))
shap.summary_plot(shap_values, X_holdout, feature_names=features, plot_type='bar', show=False)
plt.title('SHAP Feature Importance - Média Absoluta', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Bar plot gerado!")

In [0]:
# Ranking SHAP completo
shap_importance = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({
    'feature': features,
    'shap_importance': shap_importance,
    'shap_mean': shap_values.mean(axis=0),
    'shap_std': shap_values.std(axis=0)
}).sort_values('shap_importance', ascending=False).reset_index(drop=True)

print("\n" + "="*80)
print("🏆 RANKING SHAP COMPLETO")
print("="*80)
print("\nImportância = Média Absoluta | Direção = Média dos Valores SHAP\n")
print(f"{'Pos':<5} {'Feature':<25} {'Importância':<15} {'Média SHAP':<15} {'Desvio':<15}")
print("-" * 80)

for i, row in shap_df.iterrows():
    direction = "↑ positivo" if row['shap_mean'] > 0 else "↓ negativo"
    print(f"{i+1:<5} {row['feature']:<25} {row['shap_importance']:<15.6f} {row['shap_mean']:<15.6f} {row['shap_std']:<15.6f}")
    
print("\n📄 Legenda:")
print("  - Importância: Quão forte é o impacto (em módulo)")
print("  - Média SHAP: Direção média (positivo = aumenta proba, negativo = reduz)")
print("  - Desvio: Variação do impacto entre observações")

In [0]:
print("\n" + "="*80)
print("📝 EXPLICAÇÃO EM LINGUAGEM SIMPLES")
print("="*80)

# Top 5 features
top5 = shap_df.head(5)

print("\n🎯 Top 5 Features Mais Influentes:\n")
for i, row in top5.iterrows():
    direction = "aumentar" if row['shap_mean'] > 0 else "reduzir"
    print(f"{i+1}. {row['feature']}")
    print(f"   - Importância: {row['shap_importance']:.6f}")
    print(f"   - O modelo associou essa feature a {direction} a probabilidade prevista")
    print(f"   - Impacto médio: {row['shap_mean']:.6f}")
    print()

# Análise por grupo
print("\n📊 Análise por Grupo de Features:\n")

for group_name, group_features in groups.items():
    group_shap = shap_df[shap_df['feature'].isin(group_features)]
    if len(group_shap) > 0:
        total_importance = group_shap['shap_importance'].sum()
        avg_direction = group_shap['shap_mean'].mean()
        direction_text = "aumentar" if avg_direction > 0 else "reduzir"
        
        print(f"📊 {group_name}:")
        print(f"  - Importância total: {total_importance:.6f}")
        print(f"  - Direção média: {direction_text} a probabilidade")
        print(f"  - Features mais importantes:")
        for _, row in group_shap.head(2).iterrows():
            print(f"    • {row['feature']}: {row['shap_importance']:.6f}")
        print()

In [0]:
print("\n" + "="*80)
print("🔍 4. DEPENDÊNCIA DAS PRINCIPAIS FEATURES")
print("="*80)

print("\n⚠️ Gráficos de dependência mostram relações, NÃO causalidade.\n")

# Top 5 features por SHAP
top5_features = shap_df.head(5)['feature'].tolist()

for i, feature in enumerate(top5_features):
    feature_idx = features.index(feature)
    
    print(f"\n{'='*80}")
    print(f"📊 {i+1}. {feature}")
    print(f"{'='*80}")
    
    # SHAP dependence plot
    fig, ax = plt.subplots(figsize=(12, 6))
    shap.dependence_plot(
        feature_idx, 
        shap_values, 
        X_holdout, 
        feature_names=features,
        show=False,
        ax=ax
    )
    plt.title(f'SHAP Dependence Plot - {feature}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Estatísticas
    feature_values = X_holdout[:, feature_idx]
    feature_shap = shap_values[:, feature_idx]
    
    print(f"\n📊 Estatísticas:")
    print(f"  Valor mínimo: {feature_values.min():.4f}")
    print(f"  Valor máximo: {feature_values.max():.4f}")
    print(f"  Valor médio: {feature_values.mean():.4f}")
    print(f"  Impacto SHAP médio: {feature_shap.mean():.6f}")
    print(f"  Impacto SHAP min/max: [{feature_shap.min():.6f}, {feature_shap.max():.6f}]")
    
    # Detectar padrões simples
    if feature_shap.mean() > 0:
        print(f"\nℹ️ O modelo associou {feature} a AUMENTAR a probabilidade em média")
    else:
        print(f"\nℹ️ O modelo associou {feature} a REDUZIR a probabilidade em média")

In [0]:
print("\n" + "="*80)
print("🏯 5. EXPLICABILIDADE POR TICKER")
print("="*80)

print("\n⚠️ Comparação dos padrões de cada FII no modelo.\n")

# Adicionar ticker aos dados
holdout_df_with_shap = holdout_df.copy()
holdout_df_with_shap['pred_proba'] = y_pred_proba
holdout_df_with_shap['pred'] = y_pred

# Adicionar SHAP values
for i, feature in enumerate(features):
    holdout_df_with_shap[f'shap_{feature}'] = shap_values[:, i]

# Análise por ticker
tickers = sorted(holdout_df['ticker'].unique())

print("📈 SHAP Médio por Ticker (Top 10 Features):\n")

for ticker in tickers:
    ticker_mask = holdout_df_with_shap['ticker'] == ticker
    ticker_df = holdout_df_with_shap[ticker_mask]
    
    # Métricas
    ticker_roc = roc_auc_score(ticker_df['target_7d'], ticker_df['pred_proba'])
    ticker_acc = accuracy_score(ticker_df['target_7d'], ticker_df['pred'])
    
    print(f"\n{'='*80}")
    print(f"🎯 {ticker}")
    print(f"{'='*80}")
    print(f"Observações: {len(ticker_df)}")
    print(f"ROC-AUC: {ticker_roc:.4f}")
    print(f"Accuracy: {ticker_acc:.2%}")
    
    # SHAP médio por feature
    ticker_shap = {}
    for feature in features:
        ticker_shap[feature] = ticker_df[f'shap_{feature}'].mean()
    
    ticker_shap_df = pd.DataFrame([
        {'feature': k, 'shap_mean': v, 'shap_abs': abs(v)} 
        for k, v in ticker_shap.items()
    ]).sort_values('shap_abs', ascending=False)
    
    print(f"\n🏆 Top 10 Features (por impacto SHAP médio absoluto):\n")
    print(f"{'Pos':<5} {'Feature':<25} {'SHAP Médio':<15}")
    print("-" * 50)
    for i, row in ticker_shap_df.head(10).iterrows():
        direction = "↑" if row['shap_mean'] > 0 else "↓"
        print(f"{i+1:<5} {row['feature']:<25} {row['shap_mean']:<15.6f} {direction}")

In [0]:
print("\n" + "="*80)
print("🔎 6. EXPLICAÇÕES LOCAIS")
print("="*80)

print("\n⚠️ Exemplos específicos selecionados por critérios objetivos.\n")

# Criar máscara de acertos/erros
holdout_df_with_shap['correct'] = (holdout_df_with_shap['pred'] == holdout_df_with_shap['target_7d'])
holdout_df_with_shap['tp'] = (holdout_df_with_shap['pred'] == 1) & (holdout_df_with_shap['target_7d'] == 1)
holdout_df_with_shap['tn'] = (holdout_df_with_shap['pred'] == 0) & (holdout_df_with_shap['target_7d'] == 0)
holdout_df_with_shap['fp'] = (holdout_df_with_shap['pred'] == 1) & (holdout_df_with_shap['target_7d'] == 0)
holdout_df_with_shap['fn'] = (holdout_df_with_shap['pred'] == 0) & (holdout_df_with_shap['target_7d'] == 1)

# Selecionar exemplos objetivos
examples = {}

# 1. Previsão correta com alta probabilidade
examples['correct_high'] = holdout_df_with_shap[holdout_df_with_shap['correct']].nlargest(1, 'pred_proba').index[0]

# 2. Previsão incorreta com alta probabilidade
examples['incorrect_high'] = holdout_df_with_shap[~holdout_df_with_shap['correct']].nlargest(1, 'pred_proba').index[0]

# 3. Verdadeiro positivo (mais próximo de 0.5)
tp_df = holdout_df_with_shap[holdout_df_with_shap['tp']]
if len(tp_df) > 0:
    examples['tp'] = tp_df.iloc[(tp_df['pred_proba'] - 0.5).abs().argsort()[:1]].index[0]

# 4. Verdadeiro negativo (mais próximo de 0.5)
tn_df = holdout_df_with_shap[holdout_df_with_shap['tn']]
if len(tn_df) > 0:
    examples['tn'] = tn_df.iloc[(tn_df['pred_proba'] - 0.5).abs().argsort()[:1]].index[0]

# 5. Falso positivo
fp_df = holdout_df_with_shap[holdout_df_with_shap['fp']]
if len(fp_df) > 0:
    examples['fp'] = fp_df.nlargest(1, 'pred_proba').index[0]

# 6. Falso negativo
fn_df = holdout_df_with_shap[holdout_df_with_shap['fn']]
if len(fn_df) > 0:
    examples['fn'] = fn_df.nsmallest(1, 'pred_proba').index[0]

# 7. HGLG11 com alta confiança e erro
hglg_incorrect = holdout_df_with_shap[
    (holdout_df_with_shap['ticker'] == 'HGLG11') & 
    (~holdout_df_with_shap['correct'])
]
if len(hglg_incorrect) > 0:
    examples['hglg_error'] = hglg_incorrect.nlargest(1, 'pred_proba').index[0]

# 8. BTLG11 com acerto
btlg_correct = holdout_df_with_shap[
    (holdout_df_with_shap['ticker'] == 'BTLG11') & 
    (holdout_df_with_shap['correct'])
]
if len(btlg_correct) > 0:
    examples['btlg_correct'] = btlg_correct.nlargest(1, 'pred_proba').index[0]

print(f"✅ {len(examples)} exemplos selecionados")

In [0]:
# Função para explicar um exemplo
def explain_example(idx, title):
    row = holdout_df_with_shap.loc[idx]
    
    print(f"\n{'='*80}")
    print(f"{title}")
    print(f"{'='*80}")
    print(f"Data: {row['date'].date()}")
    print(f"Ticker: {row['ticker']}")
    print(f"Probabilidade prevista: {row['pred_proba']:.4f}")
    print(f"Classe prevista: {row['pred']} ({'Supera IFIX' if row['pred'] == 1 else 'Não supera IFIX'})")
    print(f"Resultado real: {row['target_7d']} ({'Superou' if row['target_7d'] == 1 else 'Não superou'})")
    print(f"Status: {'✅ ACERTO' if row['correct'] else '❌ ERRO'}")
    
    # SHAP values para este exemplo
    idx_in_holdout = holdout_df_with_shap.index.get_loc(idx)
    example_shap = shap_values[idx_in_holdout]
    
    # Top 5 contribuições positivas e negativas
    shap_contributions = pd.DataFrame({
        'feature': features,
        'value': X_holdout[idx_in_holdout],
        'shap': example_shap
    }).sort_values('shap', ascending=False)
    
    print(f"\n🔼 Top 5 Contribuições POSITIVAS (aumentaram a probabilidade):\n")
    for i, r in shap_contributions.head(5).iterrows():
        print(f"  {i+1}. {r['feature']:<25} Valor: {r['value']:>10.4f}  SHAP: {r['shap']:>10.6f}")
    
    print(f"\n🔽 Top 5 Contribuições NEGATIVAS (reduziram a probabilidade):\n")
    for i, r in shap_contributions.tail(5).iloc[::-1].iterrows():
        print(f"  {i+1}. {r['feature']:<25} Valor: {r['value']:>10.4f}  SHAP: {r['shap']:>10.6f}")
    
    # Waterfall plot
    print(f"\n💧 Waterfall Plot:\n")
    fig, ax = plt.subplots(figsize=(12, 8))
    shap.waterfall_plot(
        shap.Explanation(
            values=example_shap,
            base_values=explainer.expected_value,
            data=X_holdout[idx_in_holdout],
            feature_names=features
        ),
        show=False
    )
    plt.title(f'{title}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Explicar cada exemplo
for key, idx in examples.items():
    titles = {
        'correct_high': '🎯 Previsão Correta com Alta Probabilidade',
        'incorrect_high': '❌ Previsão Incorreta com Alta Probabilidade',
        'tp': '✅ Verdadeiro Positivo',
        'tn': '✅ Verdadeiro Negativo',
        'fp': '❌ Falso Positivo',
        'fn': '❌ Falso Negativo',
        'hglg_error': '❌ HGLG11 - Erro com Alta Confiança',
        'btlg_correct': '✅ BTLG11 - Acerto'
    }
    explain_example(idx, titles.get(key, key))

In [0]:
print("\n" + "="*80)
print("❌ 7. ANÁLISE DE ERROS")
print("="*80)

print("\n⚠️ Comparando SHAP entre acertos e erros.\n")

# Dividir em grupos
correct_mask = holdout_df_with_shap['correct']
incorrect_mask = ~correct_mask
fp_mask = holdout_df_with_shap['fp']
fn_mask = holdout_df_with_shap['fn']

print(f"📊 Distribuição:")
print(f"  Acertos: {correct_mask.sum()} ({correct_mask.mean():.2%})")
print(f"  Erros: {incorrect_mask.sum()} ({incorrect_mask.mean():.2%})")
print(f"  Falsos Positivos: {fp_mask.sum()}")
print(f"  Falsos Negativos: {fn_mask.sum()}")

# SHAP médio por grupo
groups_comparison = {}
for feature in features:
    groups_comparison[feature] = {
        'correct': holdout_df_with_shap.loc[correct_mask, f'shap_{feature}'].mean(),
        'incorrect': holdout_df_with_shap.loc[incorrect_mask, f'shap_{feature}'].mean(),
        'fp': holdout_df_with_shap.loc[fp_mask, f'shap_{feature}'].mean() if fp_mask.sum() > 0 else 0,
        'fn': holdout_df_with_shap.loc[fn_mask, f'shap_{feature}'].mean() if fn_mask.sum() > 0 else 0
    }

error_df = pd.DataFrame(groups_comparison).T
error_df['diff_correct_incorrect'] = error_df['correct'] - error_df['incorrect']
error_df['abs_diff'] = error_df['diff_correct_incorrect'].abs()
error_df = error_df.sort_values('abs_diff', ascending=False)

print(f"\n🔍 Features com maior diferença SHAP entre Acertos e Erros:\n")
print(f"{'Feature':<25} {'Acertos':<12} {'Erros':<12} {'Diferença':<12}")
print("-" * 65)
for feature, row in error_df.head(10).iterrows():
    print(f"{feature:<25} {row['correct']:>11.6f} {row['incorrect']:>11.6f} {row['diff_correct_incorrect']:>11.6f}")

print(f"\nℹ️ Interpretação:")
print("  - Features com grande diferença podem estar associadas aos erros")
print("  - Isso NÃO significa que remover essas features melhoraria o modelo")
print("  - É possível que essas features sejam importantes mas difíceis de prever")

In [0]:
# Erros por ticker
print(f"\n\n🏯 Erros por Ticker:\n")
error_by_ticker = holdout_df_with_shap.groupby('ticker').agg({
    'correct': ['sum', 'count', 'mean'],
    'pred_proba': 'mean'
}).round(4)

error_by_ticker.columns = ['_'.join(col).strip() for col in error_by_ticker.columns.values]
error_by_ticker = error_by_ticker.rename(columns={
    'correct_sum': 'acertos',
    'correct_count': 'total',
    'correct_mean': 'accuracy',
    'pred_proba_mean': 'prob_media'
})
error_by_ticker['erros'] = error_by_ticker['total'] - error_by_ticker['acertos']
error_by_ticker = error_by_ticker.sort_values('accuracy')

print(error_by_ticker[['total', 'acertos', 'erros', 'accuracy', 'prob_media']].to_string())

print(f"\nℹ️ Observações:")
worst_ticker = error_by_ticker.index[0]
best_ticker = error_by_ticker.index[-1]
print(f"  - Pior desempenho: {worst_ticker} ({error_by_ticker.loc[worst_ticker, 'accuracy']:.2%})")
print(f"  - Melhor desempenho: {best_ticker} ({error_by_ticker.loc[best_ticker, 'accuracy']:.2%})")

In [0]:
# Erros com alta confiança
print(f"\n\n⚠️ Erros com Alta Confiança (proba > 0.7 ou < 0.3):\n")

high_conf_errors = holdout_df_with_shap[
    (~holdout_df_with_shap['correct']) &
    ((holdout_df_with_shap['pred_proba'] > 0.7) | (holdout_df_with_shap['pred_proba'] < 0.3))
]

print(f"Total de erros com alta confiança: {len(high_conf_errors)}")
print(f"Percentual dos erros: {len(high_conf_errors) / incorrect_mask.sum():.2%}")

if len(high_conf_errors) > 0:
    print(f"\nDistribuição por ticker:")
    print(high_conf_errors['ticker'].value_counts().to_string())
    
    print(f"\n📊 Probabilidade média dos erros confiantes: {high_conf_errors['pred_proba'].mean():.4f}")
    
    # Features mais influentes nos erros confiantes
    high_conf_shap = {}
    for feature in features:
        high_conf_shap[feature] = high_conf_errors[f'shap_{feature}'].mean()
    
    high_conf_shap_df = pd.DataFrame([
        {'feature': k, 'shap_mean': v, 'shap_abs': abs(v)}
        for k, v in high_conf_shap.items()
    ]).sort_values('shap_abs', ascending=False)
    
    print(f"\n🔍 Top 10 features em erros confiantes:\n")
    print(high_conf_shap_df.head(10)[['feature', 'shap_mean']].to_string(index=False))

In [0]:
print("\n" + "="*80)
print("🏆 8. ANÁLISE DO RANKING TOP 1")
print("="*80)

print("\n⚠️ Investigando por que o Top 1 teve desempenho fraco.\n")

print("📊 Resultados do Top 1:")
print("  - Accuracy individual: 58.30%")
print("  - ROC-AUC: 0.6175")
print("  - Top 1 diário: 47.77%")
print("  - Top 1 não sobreposto (7 pregões): 36.11%")
print("  - Melhor FII absoluto: 22.22%")
print("\n🔍 O modelo classifica razoavelmente, mas falha no ranking.\n")

# Simular ranking Top 1 por data
holdout_df_with_shap_sorted = holdout_df_with_shap.sort_values(['date', 'pred_proba'], ascending=[True, False])

# Pegar Top 1 de cada data
top1_per_date = holdout_df_with_shap_sorted.groupby('date').first().reset_index()
top1_per_date['is_top1_correct'] = top1_per_date['target_7d'] == 1

print(f"📅 Análise diária:")
print(f"  Total de datas: {len(top1_per_date)}")
print(f"  Top 1 corretos: {top1_per_date['is_top1_correct'].sum()}")
print(f"  Taxa Top 1: {top1_per_date['is_top1_correct'].mean():.2%}")

# Ticker mais selecionado
print(f"\n🎯 Distribuição Top 1 por Ticker:\n")
top1_ticker_dist = top1_per_date['ticker'].value_counts()
for ticker, count in top1_ticker_dist.items():
    ticker_correct = top1_per_date[top1_per_date['ticker'] == ticker]['is_top1_correct'].sum()
    ticker_rate = ticker_correct / count if count > 0 else 0
    print(f"  {ticker}: {count} vezes ({count/len(top1_per_date):.1%}) - Taxa: {ticker_rate:.2%}")

In [0]:
# SHAP dos Top 1
# Pegar os índices originais do DataFrame, não do groupby
top1_original_indices = []
for date in holdout_df_with_shap['date'].unique():
    date_df = holdout_df_with_shap[holdout_df_with_shap['date'] == date]
    if len(date_df) > 0:
        top1_idx = date_df['pred_proba'].idxmax()
        top1_original_indices.append(top1_idx)

top1_shap = {}
for feature in features:
    top1_shap[feature] = holdout_df_with_shap.loc[top1_original_indices, f'shap_{feature}'].mean()

top1_shap_df = pd.DataFrame([
    {'feature': k, 'shap_mean': v, 'shap_abs': abs(v)}
    for k, v in top1_shap.items()
]).sort_values('shap_abs', ascending=False)

print(f"\n\n🔍 Features que Dominaram o Ranking Top 1:\n")
print(f"{'Pos':<5} {'Feature':<25} {'SHAP Médio':<15}")
print("-" * 50)
for i, row in top1_shap_df.head(15).iterrows():
    direction = "↑" if row['shap_mean'] > 0 else "↓"
    print(f"{i+1:<5} {row['feature']:<25} {row['shap_mean']:<15.6f} {direction}")

print(f"\nℹ️ Interpretação:")
print("  - Estas são as features que mais influenciaram os FIIs classificados como Top 1")
print("  - Se poucas features dominam, o ranking pode ser frágil")

In [0]:
# Separação de probabilidades
print(f"\n\n📊 Separação entre Probabilidades:\n")

# Para cada data, calcular diferença entre 1º e 2º colocados
separations = []
for date in holdout_df_with_shap['date'].unique():
    date_df = holdout_df_with_shap[holdout_df_with_shap['date'] == date].nlargest(2, 'pred_proba')
    if len(date_df) >= 2:
        first_proba = date_df.iloc[0]['pred_proba']
        second_proba = date_df.iloc[1]['pred_proba']
        separation = first_proba - second_proba
        separations.append(separation)

separations_series = pd.Series(separations)

print(f"  Média: {separations_series.mean():.4f}")
print(f"  Mediana: {separations_series.median():.4f}")
print(f"  Mínimo: {separations_series.min():.4f}")
print(f"  Máximo: {separations_series.max():.4f}")
print(f"  Desvio padrão: {separations_series.std():.4f}")

# Distribuição
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(separations, bins=30, edgecolor='black', alpha=0.7)
ax.axvline(separations_series.mean(), color='red', linestyle='--', linewidth=2, label=f'Média: {separations_series.mean():.4f}')
ax.set_xlabel('Diferença de Probabilidade (1º - 2º colocado)', fontsize=12)
ax.set_ylabel('Frequência', fontsize=12)
ax.set_title('Distribuição da Separação entre Top 1 e Top 2', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nℹ️ Hipótese:")
if separations_series.mean() < 0.05:
    print("  ⚠️ SEPARAÇÃO BAIXA! O 1º e 2º colocados têm probabilidades muito próximas.")
    print("  Isso explica por que o Top 1 é frágil: pequenas variações mudam o ranking.")
else:
    print("  ✅ Separação razoável entre Top 1 e Top 2.")
    print("  O problema do Top 1 pode ser falta de calibração, não separação.")

In [0]:
# Por que HGLG11 foi tão selecionado?
print(f"\n\n🔍 Por que HGLG11 foi Tão Selecionado?\n")

hglg_count = top1_ticker_dist.get('HGLG11', 0)
hglg_rate = top1_per_date[top1_per_date['ticker'] == 'HGLG11']['is_top1_correct'].mean()

print(f"🎯 HGLG11 foi selecionado {hglg_count} vezes ({hglg_count/len(top1_per_date):.1%})")
print(f"Taxa de acerto quando selecionado: {hglg_rate:.2%}")

if hglg_count > 0:
    # SHAP médio do HGLG11 quando foi Top 1
    # Pegar os índices originais do DataFrame
    hglg_top1_original_indices = []
    for date in holdout_df_with_shap[holdout_df_with_shap['ticker'] == 'HGLG11']['date'].unique():
        date_df = holdout_df_with_shap[(holdout_df_with_shap['date'] == date) & (holdout_df_with_shap['ticker'] == 'HGLG11')]
        if len(date_df) > 0:
            # Verificar se HGLG11 foi Top 1 nesta data
            all_date_df = holdout_df_with_shap[holdout_df_with_shap['date'] == date]
            top1_ticker = all_date_df.loc[all_date_df['pred_proba'].idxmax(), 'ticker']
            if top1_ticker == 'HGLG11':
                hglg_top1_original_indices.append(date_df.index[0])
    
    hglg_top1_shap = {}
    for feature in features:
        hglg_top1_shap[feature] = holdout_df_with_shap.loc[hglg_top1_original_indices, f'shap_{feature}'].mean()
    
    hglg_top1_shap_df = pd.DataFrame([
        {'feature': k, 'shap_mean': v, 'shap_abs': abs(v)}
        for k, v in hglg_top1_shap.items()
    ]).sort_values('shap_abs', ascending=False)
    
    print(f"\n🔍 Top 10 features quando HGLG11 foi Top 1:\n")
    print(hglg_top1_shap_df.head(10)[['feature', 'shap_mean']].to_string(index=False))
    
    print(f"\nℹ️ Hipótese:")
    print("  - HGLG11 pode ter valores consistentemente altos nas features importantes")
    print("  - Mas esses valores não se traduziram em outperformance real")
    print("  - Possível overfitting a padrões históricos do HGLG11 que não se repetiram")

In [0]:
print("\n" + "="*80)
print("🌍 9. VARIÁVEIS MACROECONÔMICAS")
print("="*80)

print("\n⚠️ Importância e SHAP medem influência nas previsões, NÃO causalidade.\n")

macro_analysis = {}
for macro_var in macro_features:
    # Importância nativa
    native_imp = importance_df[importance_df['feature'] == macro_var]['gain_norm'].values[0]
    
    # SHAP
    shap_imp = shap_df[shap_df['feature'] == macro_var]['shap_importance'].values[0]
    shap_mean = shap_df[shap_df['feature'] == macro_var]['shap_mean'].values[0]
    
    macro_analysis[macro_var] = {
        'native_importance': native_imp,
        'shap_importance': shap_imp,
        'shap_mean': shap_mean,
        'direction': 'aumenta' if shap_mean > 0 else 'reduz'
    }

print("📊 Análise das Variáveis Macroeconômicas:\n")
print(f"{'Variável':<15} {'Import. Nativa':>15} {'Import. SHAP':>15} {'Média SHAP':>15} {'Direção':>12}")
print("-" * 75)
for var, analysis in macro_analysis.items():
    print(f"{var:<15} {analysis['native_importance']:>14.2f}% {analysis['shap_importance']:>15.6f} "
          f"{analysis['shap_mean']:>15.6f} {analysis['direction']:>12}")

print(f"\n\nℹ️ Interpretação:")
total_macro_native = sum(a['native_importance'] for a in macro_analysis.values())
total_macro_shap = sum(a['shap_importance'] for a in macro_analysis.values())

print(f"  - Importância nativa total (macro): {total_macro_native:.2f}%")
print(f"  - Importância SHAP total (macro): {total_macro_shap:.6f}")

if total_macro_native < 10:
    print("  - ⚠️ BAIXA importância das variáveis macro")
    print("  - O modelo depende mais de features de preço/retorno/dividendo")
else:
    print("  - ✅ Variáveis macro têm importância relevante")

In [0]:
# Evolução temporal das variáveis macro no holdout
print(f"\n\n📊 Evolução das Variáveis Macro no Holdout:\n")

macro_evolution = holdout_df_with_shap.groupby('date')[macro_features].first().reset_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, macro_var in enumerate(macro_features):
    ax = axes[i]
    ax.plot(macro_evolution['date'], macro_evolution[macro_var], linewidth=2, color='steelblue')
    ax.set_title(f'{macro_var.upper()}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Data', fontsize=10)
    ax.set_ylabel('Valor', fontsize=10)
    ax.grid(alpha=0.3)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("✅ Gráficos gerados")

In [0]:
# Variação da contribuição SHAP ao longo do tempo
print(f"\n\n📊 Variação da Contribuição SHAP ao Longo do Holdout:\n")

for macro_var in macro_features:
    shap_col = f'shap_{macro_var}'
    monthly_shap = holdout_df_with_shap.groupby(holdout_df_with_shap['date'].dt.to_period('M'))[shap_col].mean()
    
    print(f"\n{macro_var.upper()}:")
    print(f"  Média SHAP por mês:\n")
    for period, value in monthly_shap.items():
        direction = "↑" if value > 0 else "↓"
        print(f"    {period}: {value:>10.6f} {direction}")

In [0]:
# Diferenças por ticker
print(f"\n\n🏯 Diferenças por Ticker (SHAP Médio das Macro):\n")

for ticker in tickers:
    ticker_mask = holdout_df_with_shap['ticker'] == ticker
    print(f"\n{ticker}:")
    for macro_var in macro_features:
        shap_col = f'shap_{macro_var}'
        ticker_shap = holdout_df_with_shap.loc[ticker_mask, shap_col].mean()
        direction = "↑" if ticker_shap > 0 else "↓"
        print(f"  {macro_var:<15} {ticker_shap:>10.6f} {direction}")

## ⚠️ 10. LIMITAÇÕES DA EXPLICABILIDADE

---

### Princípios Fundamentais

1. **SHAP explica o modelo, NÃO a realidade econômica**
   - Valores SHAP mostram como o modelo usa as features
   - Isso NÃO prova que essas features causam o resultado
   - Um modelo pode aprender correlações espúrias

2. **Correlação ≠ Causalidade**
   - Uma feature importante pode ser apenas um proxy
   - Variáveis correlacionadas dividem importância
   - A ordem temporal não garante causalidade

3. **Features correlacionadas dividem importância**
   - return_7d e return_30d são correlacionados
   - SHAP pode distribuir importância entre eles
   - Uma feature sozinha pode parecer menos importante do que realmente é

4. **Dependência de dados**
   - Resultados são válidos para:
     - Período: 2025-03-01 a 2026-02-28
     - Tickers: BTLG11, HGLG11, LVBI11, VILG11, XPLG11
   - Outros períodos/tickers podem ter padrões diferentes

5. **Explicações locais não garantem generalização**
   - Um exemplo individual pode não representar o padrão geral
   - Explicações diferentes para observações similares são possíveis

6. **Importância ≠ Utilidade para retreino**
   - Uma feature importante pode:
     - Ser difícil de prever corretamente
     - Capturar ruído em vez de sinal
     - Não melhorar performance fora da amostra
   - Remover features importantes pode piorar OU melhorar o modelo

7. **O modelo final está CONGELADO**
   - Este notebook apenas explica
   - NãO deve ser usado para retreinar
   - Explicações não implicam melhorias automáticas

---

### Cuidados ao Interpretar

* Use linguagem cautelosa:
  - ✅ "O modelo associou X a aumentar a probabilidade"
  - ❌ "X causa aumento no retorno"

* Não generalize além dos dados:
  - ✅ "No holdout, dividendos tiveram baixa importância"
  - ❌ "Dividendos não importam para FIIs"

* Reconheça incerteza:
  - ✅ "O padrão sugere que..."
  - ❌ "Está provado que..."

---

In [0]:
print("\n" + "="*80)
print("📝 11. CONCLUSÃO EXECUTIVA")
print("="*80)

print("\n" + "="*80)
print("RESPOSTAS ÀS QUESTÕES-CHAVE")
print("="*80)

# 1. Top 5 features
print("\n\n1️⃣ Quais foram as 5 features mais importantes?\n")
top5 = shap_df.head(5)
for i, row in top5.iterrows():
    native = importance_df[importance_df['feature'] == row['feature']]['gain_norm'].values[0]
    print(f"   {i+1}. {row['feature']}")
    print(f"      - Importância SHAP: {row['shap_importance']:.6f}")
    print(f"      - Importância Nativa: {native:.2f}%")
    print(f"      - Direção média: {('aumenta' if row['shap_mean'] > 0 else 'reduz')} probabilidade")
    print()

# 2. Fatores que aumentaram chance
print("\n2️⃣ Quais fatores normalmente AUMENTARAM a chance de superar o IFIX?\n")
positive_features = shap_df[shap_df['shap_mean'] > 0].head(10)
for i, row in positive_features.iterrows():
    print(f"   • {row['feature']}: SHAP médio = {row['shap_mean']:.6f}")

# 3. Fatores que reduziram chance
print("\n\n3️⃣ Quais fatores normalmente REDUZIRAM a chance de superar o IFIX?\n")
negative_features = shap_df[shap_df['shap_mean'] < 0].head(10)
for i, row in negative_features.iterrows():
    print(f"   • {row['feature']}: SHAP médio = {row['shap_mean']:.6f}")

# 4. Influência macro
print("\n\n4️⃣ As variáveis macro tiveram influência relevante?\n")
total_macro_importance = sum(macro_analysis[var]['shap_importance'] for var in macro_features)
total_importance = shap_df['shap_importance'].sum()
macro_percentage = (total_macro_importance / total_importance) * 100

print(f"   📊 Importância SHAP total das macros: {total_macro_importance:.6f}")
print(f"   📊 Percentual do total: {macro_percentage:.2f}%")
print()
for var in macro_features:
    print(f"   • {var}: {macro_analysis[var]['shap_importance']:.6f} ({macro_analysis[var]['direction']} proba)")

if macro_percentage < 10:
    print(f"\n   ⚠️ BAIXA influência: Macro representou apenas {macro_percentage:.1f}%")
    print("   O modelo depende principalmente de preços, retornos e dividendos.")
elif macro_percentage < 20:
    print(f"\n   ℹ️ MODERADA influência: Macro representou {macro_percentage:.1f}%")
    print("   Variáveis macro têm papel secundário mas não desprezível.")
else:
    print(f"\n   ✅ ALTA influência: Macro representou {macro_percentage:.1f}%")
    print("   Variáveis macro são componente importante do modelo.")

In [0]:
# 5. Plausibilidade econômica
print("\n\n5️⃣ O comportamento das features foi economicamente plausível?\n")

print("   ✅ SIM, em sua maioria:")
print("   • Retornos recentes positivos associados a maior probabilidade: plausível (momentum)")
print("   • Volatilidade pode ter relação ambígua: plausível (risco vs oportunidade)")
print("   • Dividend yield alto pode aumentar atratividade: plausível")
print("\n   ⚠️ MAS ATENÇÃO:")
print("   • Plausibilidade NÃO prova que o modelo capturou causalidade")
print("   • O modelo pode estar aprendendo correlações espúrias")
print("   • Padrões que parecem lógicos ex-post podem ser coincidência")

# 6. Desempenho fraco do Top 1
print("\n\n6️⃣ O que pode explicar o desempenho fraco do Top 1?\n")

print("   📉 Resultados:")
print(f"   • Accuracy individual: 58.30%")
print(f"   • Top 1 diário: 47.77%")
print(f"   • Top 1 não sobreposto: 36.11%")
print(f"   • Melhor FII absoluto: 22.22%")

print("\n   🔍 Hipóteses Identificadas:")
print("\n   1. SEPARAÇÃO BAIXA entre probabilidades:")
if 'separations_series' in locals():
    print(f"      • Diferença média entre 1º e 2º: {separations_series.mean():.4f}")
    if separations_series.mean() < 0.05:
        print(f"      • ⚠️ Separação muito baixa! Ranking é frágil.")

print("\n   2. HGLG11 DOMINANDO o ranking:")
if 'top1_ticker_dist' in locals() and 'HGLG11' in top1_ticker_dist.index:
    hglg_pct = top1_ticker_dist['HGLG11'] / top1_ticker_dist.sum()
    print(f"      • HGLG11 foi Top 1 em {hglg_pct:.1%} das vezes")
    if 'hglg_rate' in locals():
        print(f"      • Taxa de acerto: apenas {hglg_rate:.2%}")
    print(f"      • ⚠️ Modelo pode estar enviesado para HGLG11")

print("\n   3. CALIBRAÇÃO INADEQUADA:")
print("      • ROC-AUC 0.6175 mostra separação moderada de classes")
print("      • Mas as probabilidades não estão bem calibradas para ranking")
print("      • Modelo bom em classificar, fraco em ordenar")

print("\n   4. OVERFITTING A PADRÕES DE CURTO PRAZO:")
print("      • Holdout diário: 47.77%")
print("      • Holdout não sobreposto (7 dias): 36.11%")
print("      • ⚠️ Queda de 11.66 p.p. sugere dependência temporal")

In [0]:
# 7. Erros do HGLG11
print("\n\n7️⃣ O que pode explicar os erros frequentes do HGLG11?\n")

if 'error_by_ticker' in locals() and 'HGLG11' in error_by_ticker.index:
    hglg_acc = error_by_ticker.loc['HGLG11', 'accuracy']
    print(f"   📉 Accuracy do HGLG11: {hglg_acc:.2%}")
    
    if hglg_acc < 0.50:
        print(f"   ⚠️ ABAIXO de 50%! Pior que aleatório.")
    elif hglg_acc < accuracy:
        print(f"   ⚠️ Abaixo da média geral ({accuracy:.2%})")

print("\n   🔍 Hipóteses:")
print("   1. HGLG11 pode ter características diferentes dos outros FIIs")
print("   2. Modelo foi treinado com todos os FIIs, mas HGLG11 é mais difícil de prever")
print("   3. HGLG11 pode ter mudado de regime entre treino e holdout")
print("   4. Features que funcionam bem para outros FIIs falham para HGLG11")
print("   5. HGLG11 aparece muito como Top 1 mas não supera IFIX na prática")

# 8. Dependência excessiva
print("\n\n8️⃣ O modelo depende excessivamente de alguma feature?\n")

top1_feature = shap_df.iloc[0]
top1_importance = top1_feature['shap_importance']
total_importance = shap_df['shap_importance'].sum()
top1_percentage = (top1_importance / total_importance) * 100

print(f"   🏆 Feature mais importante: {top1_feature['feature']}")
print(f"   📊 Importância: {top1_importance:.6f} ({top1_percentage:.1f}% do total)")

top3_importance = shap_df.head(3)['shap_importance'].sum()
top3_percentage = (top3_importance / total_importance) * 100

print(f"\n   📊 Top 3 features juntas: {top3_percentage:.1f}% do total")

if top1_percentage > 30:
    print(f"\n   ⚠️ ALTA dependência de {top1_feature['feature']}!")
    print("   Risco: se essa feature for ruidosa, todo o modelo sofre")
elif top3_percentage > 50:
    print(f"\n   ⚠️ Top 3 features dominam o modelo ({top3_percentage:.1f}%)")
    print("   O modelo é concentrado, não diversificado")
else:
    print(f"\n   ✅ Dependência distribuída: Top 1 = {top1_percentage:.1f}%, Top 3 = {top3_percentage:.1f}%")
    print("   O modelo usa múltiplas features de forma equilibrada")

In [0]:
# 9. Confiança no classificador
print("\n\n9️⃣ As explicações aumentam a confiança no classificador individual?\n")

print("   ✅ SIM, em aspectos:")
print("   • Features mais importantes fazem sentido econômico")
print("   • Modelo usa múltiplas fontes de informação (preço, dividendo, macro)")
print("   • Explicações locais são coerentes com explicações globais")
print(f"   • ROC-AUC 0.6175 e Accuracy 58.30% são consistentes")

print("\n   ⚠️ MAS com ressalvas:")
print("   • Plausibilidade NÃO garante que o modelo capturou relações verdadeiras")
print("   • Desempenho fraco no Top 1 indica problema de calibração/ranking")
print("   • Diferenças grandes entre tickers (HGLG11 vs BTLG11) preocupam")
print("   • Queda de performance em datas não sobrepostas sugere fragilidade temporal")

print("\n   🎯 Veredito:")
print("   O classificador individual é RAZOÁVEL mas NÃO EXCEPCIONAL.")
print("   Pode ser usado com cautela, mas NÃO para ranking Top 1 sozinho.")

# 10. Cuidados para documentação
print("\n\n🔟 O que mencionar no README e LinkedIn?\n")

print("\n   📝 Pontos OBRIGATÓRIOS a mencionar:")
print("\n   1. CORRELAÇÃO ≠ CAUSALIDADE")
print("      • 'O modelo associou X a Y'")
print("      • NÃO 'X causa Y'")

print("\n   2. LIMITAÇÕES TEMPORAIS")
print("      • Período: 2025-03-01 a 2026-02-28")
print("      • Padrões podem mudar em outros períodos")

print("\n   3. LIMITAÇÕES DE TICKERS")
print("      • Apenas 5 FIIs: BTLG11, HGLG11, LVBI11, VILG11, XPLG11")
print("      • NãO generalizar para todos os FIIs")

print("\n   4. DESEMPENHO DO TOP 1")
print("      • Classificador individual: 58.30%")
print("      • Top 1 diário: 47.77%")
print("      • Top 1 não sobreposto: 36.11%")
print("      • ⚠️ Modelo NÃO deve ser usado para ranking direto")

print("\n   5. NATURE EXPLORATÓRIA")
print("      • Projeto é acadêmico/exploratório")
print("      • NÃO é recomendação de investimento")
print("      • NÃO usar para decisões financeiras reais")

print("\n   6. EXPLICABILIDADE DESCRITIVA")
print("      • SHAP explica o MODELO, não a realidade")
print("      • Features importantes podem ser proxies")
print("      • Importância NÃO garante utilidade para melhoria")

In [0]:
print("\n\n" + "="*80)
print("🏁 RESUMO EXECUTIVO FINAL")
print("="*80)

print("\n👍 PONTOS FORTES:")
print("  1. Features mais importantes fazem sentido econômico")
print("  2. Modelo usa múltiplas fontes de informação")
print("  3. Desempenho de classificação é razoável (58.30%)")
print("  4. Explicações são consistentes entre níveis (global/local)")
print("  5. Não há dependência excessiva de uma única feature")

print("\n⚠️ PONTOS FRACOS:")
print("  1. Top 1 tem desempenho fraco (36.11% não sobreposto)")
print("  2. Separação baixa entre probabilidades do ranking")
print("  3. HGLG11 domina ranking mas tem baixa taxa de acerto")
print("  4. Performance cai em datas não sobrepostas (fragilidade temporal)")
print("  5. Variáveis macro têm importância baixa")
print("  6. Diferenças grandes de desempenho entre tickers")

print("\n🎯 RECOMENDAÇÕES:")
print("  1. ✅ USAR o classificador individual com cautela")
print("  2. ❌ NÃO USAR para ranking Top 1 direto")
print("  3. 🔍 CONSIDERAR métodos de ensemble ou calibração para ranking")
print("  4. 🔍 INVESTIGAR por que HGLG11 tem padrões diferentes")
print("  5. 🔍 VALIDAR em período futuro antes de uso real")

print("\n📚 DOCUMENTAÇÃO:")
print("  • Mencionar explicitamente: correlação ≠ causalidade")
print("  • Destacar limitações: período, tickers, Top 1")
print("  • Reforçar natureza exploratória: NÃO é recomendação")
print("  • Explicar que SHAP descreve o modelo, não a realidade")

print("\n" + "="*80)
print("✅ ANÁLISE DE EXPLICABILIDADE CONCLUÍDA")
print("="*80)

print("\n📝 O modelo foi explicado com sucesso usando:")
print("  • Importância nativa do XGBoost (gain, weight, cover)")
print("  • SHAP TreeExplainer para explicabilidade global e local")
print("  • Análise por ticker")
print("  • Investigação de erros e do problema do Top 1")
print("  • Avaliação de variáveis macroeconômicas")

print("\n⚠️ Lembre-se:")
print("  Este notebook apenas EXPLICA o modelo final.")
print("  NÃO deve ser usado para retreinar ou modificar o modelo.")
print("  Explicações não implicam melhorias automáticas.")

print("\n" + "="*80)